In [0]:
# Questo notebook inizializza la tabella Silver energia del progetto.
#
# Legge tutti i dati ENTSO-E già caricati nella tabella Bronze:
# progetto_meteo.bronze.dati_entsoe
#
# Poi li aggrega per:
# - Area
# - Data
# - Ora
#
# Il risultato viene salvato nella tabella Silver:
# progetto_meteo.silver.energy_block
#
# Questa tabella contiene le tre grandezze energetiche finali usate dal progetto:
# - Solare
# - Idrico
# - Eolico
#
# Questo notebook serve nella fase iniziale, subito dopo Bootstrap_entsoe_API.
# Per gli aggiornamenti giornalieri, invece, si usa Patcher_Energy_Block,
# che aggiorna solo il giorno appena concluso.
#
# La logica di calcolo resta semplice:
# - Solare deriva da Solar
# - Idrico deriva da Hydro Run-of-river and pondage + Hydro Water Reservoir
# - Eolico deriva da Wind Offshore + Wind Onshore

from pyspark.sql import functions as F


# Imposto catalogo e tabelle del progetto.
catalogo = "progetto_meteo"

tabella_entsoe = f"{catalogo}.bronze.dati_entsoe"
tabella_energy_block = f"{catalogo}.silver.energy_block"


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Leggo tutti i dati ENTSO-E presenti nella Bronze.
# Questa tabella è stata popolata dal notebook Bootstrap_entsoe_API.
df_entsoe = spark.table(tabella_entsoe)


# Controllo che la Bronze abbia dati.
# Se la tabella è vuota, interrompo il notebook perché non avrebbe senso creare la Silver energia.
righe_bronze = df_entsoe.count()

if righe_bronze == 0:
    raise Exception(f"Nessun dato trovato in {tabella_entsoe}. Esegui prima Bootstrap_entsoe_API.")


# Aggrego tutta la Bronze ENTSO-E per Area, Data e Ora.
# In questa fase porto i tipi di produzione tecnici ENTSO-E
# nelle tre colonne finali del progetto: Solare, Idrico ed Eolico.
df_energy_block = (
    df_entsoe
    .groupBy(
        "Area",
        "Data",
        "Ora"
    )
    .agg(
        F.round(
            F.sum(
                F.when(
                    F.col("Tipo_Produzione") == "Solar",
                    F.col("Produzione")
                )
            ),
            2
        ).alias("Solare"),

        F.round(
            F.sum(
                F.when(
                    F.col("Tipo_Produzione").isin(
                        "Hydro Run-of-river and pondage",
                        "Hydro Water Reservoir"
                    ),
                    F.col("Produzione")
                )
            ),
            2
        ).alias("Idrico"),

        F.round(
            F.sum(
                F.when(
                    F.col("Tipo_Produzione").isin(
                        "Wind Offshore",
                        "Wind Onshore"
                    ),
                    F.col("Produzione")
                )
            ),
            2
        ).alias("Eolico")
    )
    .fillna({
        "Solare": 0.0,
        "Idrico": 0.0,
        "Eolico": 0.0
    })
    .withColumn("Data", F.date_format("Data", "dd/MM/yyyy"))
    .select(
        "Area",
        "Data",
        "Ora",
        "Solare",
        "Idrico",
        "Eolico"
    )
)


# Controllo che l'aggregazione abbia prodotto righe.
# Se non ci sono righe, non aggiorno energy_block.
righe_energy_block = df_energy_block.count()

if righe_energy_block == 0:
    raise Exception("Aggregazione ENTSO-E completata, ma nessuna riga prodotta. Non aggiorno energy_block.")


# Sovrascrivo tutta la tabella Silver energy_block.
# Uso overwrite perché questo notebook serve a inizializzare la Silver energia
# partendo da tutta la Bronze ENTSO-E.
df_energy_block.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(
    tabella_energy_block
)


# Controllo finale.
# Mostro un campione ordinato della Silver energia appena creata.
display(
    spark.table(tabella_energy_block)
    .orderBy("Area", "Data", "Ora")
    .limit(100)
)


# Stampo un riepilogo finale della clonazione energia.
print(f"Clonazione Energy Block completata: {tabella_entsoe} → {tabella_energy_block}")
print(f"Righe Bronze ENTSO-E lette: {righe_bronze}")
print(f"Righe copiate in energy_block: {spark.table(tabella_energy_block).count()}")